# MIF 2026 — verificação reproduzível da fixture de canais

## tl;dr

Este notebook executa o pipeline de produção sobre uma **fixture sintética de desenvolvimento**. Ele valida 31 queries, reconciliação, cobertura de mapeamento, spot checks de manchetes e ausência de PII nos outputs. Os resultados abaixo **não são resultados finais de 2026**; a Task 8 substituirá a fixture por extração fresca e mapeamentos completos.


## Context & Methods

O leitor é quem revisa a reprodutibilidade e a fronteira de privacidade da Task 7. O notebook usa os mesmos módulos de fonte, facts, mapeamento, métricas, privacidade, narrativa e snapshot empregados pela CLI.

### Key Assumptions

- evento de desenvolvimento: `72611`;
- somente status pago entra nas métricas;
- mapeamentos da fixture são explicitamente revisados;
- pedidos tocados não são aditivos entre canais;
- nenhuma inferência final sobre 2026 é permitida nesta etapa.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from _codex.analyses.mif_2026_channels.pipeline import run_analysis
from _codex.tests.mif_2026_channels.fixtures import write_study_fixture

workspace = TemporaryDirectory()
fixture_root = Path(workspace.name)
orders, participants, channel_map, product_map = write_study_fixture(fixture_root)
outputs = run_analysis(orders, participants, channel_map, product_map, fixture_root / "report_app", allow_stale=True)
report = json.loads(outputs["report_data"].read_text(encoding="utf-8"))
aggregates = json.loads(outputs["aggregates"].read_text(encoding="utf-8"))
reconciliation = json.loads(outputs["reconciliation"].read_text(encoding="utf-8"))
source_notes = json.loads(outputs["source_notes"].read_text(encoding="utf-8"))
{"status": report["status"], "queries": len(report["queries"]), "surface": report["surface"]}


{'status': 'fixture', 'queries': 31, 'surface': 'report'}

## Data

A fixture contém canais com tratamento completo e cauda longa para exercitar o relatório. A tabela de qualidade abaixo vem do output de produção, não de cálculo copiado no notebook.


In [2]:
quality_preview = aggregates["datasets"]["data_quality"][:6]
quality_preview


[{'coverage_pct': 100.0, 'denominator': 26, 'field': 'cashback_value', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}, {'coverage_pct': 100.0, 'denominator': 26, 'field': 'declared_registration_count', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}, {'coverage_pct': 100.0, 'denominator': 26, 'field': 'device_type', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}, {'coverage_pct': 100.0, 'denominator': 26, 'field': 'discount_value', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}, {'coverage_pct': 100.0, 'denominator': 26, 'field': 'fee_value', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}, {'coverage_pct': 100.0, 'denominator': 26, 'field': 'gross_order_value', 'grain': 'order', 'invalid': 0, 'missing': 0, 'valid': 26}]

In [3]:
mapping_coverage = {
    "channel_mapping_coverage_pct": reconciliation["channel_mapping_coverage_pct"],
    "product_mapping_coverage_pct": reconciliation["product_mapping_coverage_pct"],
}
assert mapping_coverage == {"channel_mapping_coverage_pct": 100.0, "product_mapping_coverage_pct": 100.0}
mapping_coverage


{'channel_mapping_coverage_pct': 100.0, 'product_mapping_coverage_pct': 100.0}

## Results

### 1. Reconcile additive totals


In [4]:
overview = aggregates["overview"]
datasets = aggregates["datasets"]
checks = {
    "receipt_vs_overview": reconciliation["paid_registration_count"] == overview["paid_registrations"],
    "modalities_vs_overview": sum(row["paid_registrations"] for row in datasets["modality_mix"]) == overview["paid_registrations"],
    "lots_vs_overview": sum(row["paid_registrations"] for row in datasets["lot_performance"]) == overview["paid_registrations"],
    "channels_vs_overview": sum(row["paid_registrations"] for row in datasets["channel_index"]) == overview["paid_registrations"],
}
assert all(checks.values())
checks


{'receipt_vs_overview': True, 'modalities_vs_overview': True, 'lots_vs_overview': True, 'channels_vs_overview': True}

### 2. Spot-check fixture headlines

Os valores abaixo servem apenas como sentinelas reprodutíveis da fixture.


In [5]:
headline_checks = {
    "paid_orders": overview["paid_orders"],
    "paid_registrations": overview["paid_registrations"],
    "gross_value": overview["gross_value"],
    "full_dossiers": [row["channel_name"] for row in aggregates["full_dossiers"]],
    "long_tail": [row["channel_name"] for row in aggregates["long_tail"]],
}
assert headline_checks == {
    "paid_orders": 26, "paid_registrations": 26, "gross_value": "2780.00",
    "full_dossiers": ["Canal A", "Canal B"], "long_tail": ["Canal C"],
}
headline_checks


{'paid_orders': 26, 'paid_registrations': 26, 'gross_value': '2780.00', 'full_dossiers': ['Canal A', 'Canal B'], 'long_tail': ['Canal C']}

### 3. Check the reviewed-data privacy boundary


In [6]:
combined = "\n".join(output.read_text(encoding="utf-8") for output in outputs.values()).lower()
forbidden_tokens = ["numero_pedido", "numero_inscricao", '"facts"', "example.test", "segredo-teste"]
privacy_checks = {token: token not in combined for token in forbidden_tokens}
assert all(privacy_checks.values())
privacy_checks


{'numero_pedido': True, 'numero_inscricao': True, '"facts"': True, 'example.test': True, 'segredo-teste': True}

## Takeaways

- O pipeline de produção gerou o snapshot canônico com `surface=report`, status `fixture` e 31 queries.
- Pedidos, inscrições, modalidade, lote e canais reconciliaram na fixture.
- As coberturas de mapeamento sintético estão completas para este teste.
- Os quatro outputs passaram pelos checks locais de fronteira sem facts brutos nem identificadores.
- Esta evidência valida a mecânica da Task 7, não o desempenho comercial real da Maratona de Floripa 2026.
